In [2]:
import pandas as pd
import numpy as np
import chromadb

# Load clean data
df = pd.read_csv("../data/clean_articles.csv", encoding='utf-8-sig')

# Load saved embeddings
embeddings = np.load("../data/embeddings.npy")

print("✅ Data loaded!")
print("Articles:", len(df))
print("Embeddings shape:", embeddings.shape)

✅ Data loaded!
Articles: 111860
Embeddings shape: (111860, 384)


In [3]:
# Reconnect to existing ChromaDB
client = chromadb.PersistentClient(path="../data/chromadb")

# Delete collection if already exists (fresh start)
try:
    client.delete_collection("urdu_news")
    print("Old collection deleted")
except:
    pass

# Create new collection
collection = client.create_collection(
    name="urdu_news",
    metadata={"hnsw:space": "cosine"}
)

print("✅ ChromaDB collection created!")

Old collection deleted
✅ ChromaDB collection created!


In [4]:
import time

print("Storing all articles in ChromaDB...")
print("This will take 10-15 minutes. Please wait...\n")

start_time = time.time()
batch_size = 1000

for i in range(0, len(df), batch_size):
    batch_end = min(i + batch_size, len(df))
    
    # Prepare batch data
    batch_ids = [str(j) for j in range(i, batch_end)]
    batch_embeddings = embeddings[i:batch_end].tolist()
    batch_documents = df['combined_text'].iloc[i:batch_end].tolist()
    batch_metadatas = [
        {
            "headline": str(df['Headline'].iloc[j]),
            "category": str(df['Category'].iloc[j]),
            "date": str(df['Date'].iloc[j]),
            "source": str(df['Source'].iloc[j])
        }
        for j in range(i, batch_end)
    ]
    
    # Store in ChromaDB
    collection.add(
        ids=batch_ids,
        embeddings=batch_embeddings,
        documents=batch_documents,
        metadatas=batch_metadatas
    )
    
    # Progress update every 10k articles
    if (i + batch_size) % 10000 == 0:
        print(f"Stored {min(i + batch_size, len(df)):,} / {len(df):,} articles...")

elapsed = (time.time() - start_time) / 60
print(f"\n✅ All articles stored!")
print(f"Time taken: {elapsed:.1f} minutes")
print(f"Total in DB: {collection.count()}")

Storing all articles in ChromaDB...
This will take 10-15 minutes. Please wait...

Stored 10,000 / 111,860 articles...
Stored 20,000 / 111,860 articles...
Stored 30,000 / 111,860 articles...
Stored 40,000 / 111,860 articles...
Stored 50,000 / 111,860 articles...
Stored 60,000 / 111,860 articles...
Stored 70,000 / 111,860 articles...
Stored 80,000 / 111,860 articles...
Stored 90,000 / 111,860 articles...
Stored 100,000 / 111,860 articles...
Stored 110,000 / 111,860 articles...

✅ All articles stored!
Time taken: 14.3 minutes
Total in DB: 111860


In [5]:
# Verify database is working correctly
print("Database verification:")
print("Total documents:", collection.count())

# Test a simple query
test_query = embeddings[0].tolist()

results = collection.query(
    query_embeddings=[test_query],
    n_results=5
)

print("\n✅ Query test successful!")
print("Top 5 results for first article:")
for i, metadata in enumerate(results['metadatas'][0]):
    print(f"\n{i+1}. {metadata['headline'][:60]}")
    print(f"   Category: {metadata['category']}")

Database verification:
Total documents: 111860

✅ Query test successful!
Top 5 results for first article:

1. عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا
   Category: Business & Economics

2. عالمی بینک کی جانب سے پاکستان کے لئے چھ کروڑسولہ لاکھ ڈالر ق
   Category: Business & Economics

3. ورلڈ بینک پاکستان میں غریب خواتین کو مستحکم کرنے کیلئے پرعزم
   Category: Business & Economics

4. ورلڈ بینک نے فاٹا کے متاثرین کیلئے 75ملین ڈالرامداد کی منظور
   Category: Business & Economics

5. عالمی بینک کی پاکستان کیلئے 20 کروڑ ڈالر کی ہنگامی امداد کی 
   Category: Business & Economics
